## Nivel 2: Fechas 

Las fechas aportan un valor analítico muy importante porque revelan tendencias, estacionalidad y cohortes, sin embargo, suelen ser los datos más problemas en su formato.

Para este nivel haremos una muestra ya que necesitamos emplear operaciones mas complejas que pueden agotar el kernel, ya que la memoria RAM requiere mucho esfuerzo para ejecutar operaciones para el data set completo.

Para este data set tenemos started_at y ended_at y haremos el EDA en tres momentos

- Data Profiling (Validación estructural de fechas): formato consistente, longitud, rango, nulos, duplicados esperables
- Type Casting Conversión técnica de String a Datetime64.
- Business Rules Validation (Coherencia temporal): inicio ≤ fin, duración válida, detección de outliers

In [1]:
#muestra

import pandas as pd 

df=pd.read_csv('dataset_maestro_ciclistas.csv')
df_muestra = df.sample(n=100000, random_state=42)

In [2]:
#df_muestra.to_csv('muestra_ciclistas.csv', index=False)

In [3]:
#df_backup = pd.read_csv('muestra_ciclistas.csv')

#df_muestra['end_lat'] = df_backup['end_lat']
#df_muestra = df_muestra.drop(columns=['end_lat', 'end_lng'])



In [4]:
df_muestra.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
2021231,AC09C4230F16DC5B,electric_bike,2025-06-30 19:16:08.604,2025-06-30 19:21:26.124,Wolcott Ave & Fargo Ave,CHI01266,Elmwood Ave & Austin St,CHI00739,42.016977,-87.677725,42.025784,-87.684107,casual
1103055,A38E8273171D67F0,electric_bike,2025-05-18 12:36:15.975,2025-05-18 12:55:40.536,NaN,NaN,Montrose Harbor,TA1308000012,41.900000,-87.620000,41.963982,-87.638181,member
1834169,0B1F607D321397B0,electric_bike,2025-06-17 20:43:05.503,2025-06-17 21:04:51.961,NaN,NaN,Michigan Ave & Oak St,CHI00252,41.950000,-87.650000,41.900960,-87.623777,member
277055,21649CA95A4A10B4,electric_bike,2025-02-02 05:04:52.086,2025-02-02 05:08:16.654,NaN,NaN,Clark St & Newport St,632,41.950000,-87.670000,41.944540,-87.654678,casual
740603,D6CB6A805FC24CD2,classic_bike,2025-04-07 10:24:51.016,2025-04-07 10:31:27.276,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member


In [5]:
df_muestra.shape

(100000, 13)

In [6]:
df_muestra[['started_at','ended_at']].head(10)

,started_at,ended_at
2021231,2025-06-30 19:16:08.604,2025-06-30 19:21:26.124
1103055,2025-05-18 12:36:15.975,2025-05-18 12:55:40.536
1834169,2025-06-17 20:43:05.503,2025-06-17 21:04:51.961
277055,2025-02-02 05:04:52.086,2025-02-02 05:08:16.654
740603,2025-04-07 10:24:51.016,2025-04-07 10:31:27.276
1718938,2025-06-26 20:38:46.166,2025-06-26 21:08:31.115
396011,2025-03-03 08:06:48.805,2025-03-03 08:22:54.075
3747332,2025-09-28 08:51:17.999,2025-09-28 09:22:02.975
1960416,2025-06-28 16:18:25.851,2025-06-28 16:51:47.192
3198153,2025-08-11 08:15:48.662,2025-08-11 08:49:14.477


In [7]:
#tipos de datos de las columnas de fecha y hora

df_muestra['ended_at'].apply(type).value_counts()
df_muestra['started_at'].apply(type).value_counts()

#Acción pendiente: convertir las columnas de fecha y hora a formato datetime

started_at
<class 'str'>    100000
Name: count, dtype: int64

In [8]:
#Longitud de los valores en 'started_at' y 'ended_at'

df_muestra['started_at'].astype('str').str.len().value_counts()
df_muestra['ended_at'].astype('str').str.len().value_counts()

ended_at
23    100000
Name: count, dtype: int64

In [9]:
# Inspección visual de los separadores únicos en 'started_at' y 'ended_at' 

print(df_muestra['started_at'].replace(r'\d',' ', regex=True).unique())
print(df_muestra['ended_at'].replace(r'\d',' ', regex=True).unique())

['    -  -     :  :  .   ']
['    -  -     :  :  .   ']


In [10]:
#Verificar que el rango de fechas en 'started_at' y 'ended_at' sea coherente y no contenga valores atípicos o inconsistentes.

print(df_muestra[['started_at','ended_at']].max())
print(df_muestra[['started_at','ended_at']].min())

started_at    2025-12-31 23:46:07.383
ended_at      2025-12-31 23:56:50.685
dtype: object
started_at    2024-12-31 23:42:36.959
ended_at      2025-01-01 00:16:16.440
dtype: object


In [11]:
#Extraer componentes de fecha y hora de 'started_at' para comprbar que corresponden a AAA/MM/DD HH:MM:SS    

partes_fecha=df_muestra['started_at'].str.split('-|:| ', expand=True)
partes_fecha.head()

,0,1,2,3,4,5
2021231,2025,06,30,19,16,08.604
1103055,2025,05,18,12,36,15.975
1834169,2025,06,17,20,43,05.503
277055,2025,02,02,05,04,52.086
740603,2025,04,07,10,24,51.016


In [12]:
# Extracción de año, mes y día de 'started_at' para saber que valores únicos hay

partes_fecha_s=df_muestra['started_at'].str.split('-|:| ', expand=True)
año_s=partes_fecha_s[0].unique()
mes_s=partes_fecha_s[1].unique()
dia_s=sorted(partes_fecha_s[2].unique())

año_s, mes_s, dia_s

(array(['2025', '2024'], dtype=object),
 array(['06', '05', '02', '04', '03', '09', '08', '07', '01', '11', '10',
        '12'], dtype=object),
 ['01',
  '02',
  '03',
  '04',
  '05',
  '06',
  '07',
  '08',
  '09',
  '10',
  '11',
  '12',
  '13',
  '14',
  '15',
  '16',
  '17',
  '18',
  '19',
  '20',
  '21',
  '22',
  '23',
  '24',
  '25',
  '26',
  '27',
  '28',
  '29',
  '30',
  '31'])

In [13]:
# Extracción de año, mes y día de 'ended_at' para saber que valores únicos hay

partes_fecha_e=df_muestra['ended_at'].str.split('-|:| ', expand=True)
año_e=partes_fecha_e[0].unique()
mes_e=partes_fecha_e[1].unique()
dia_e=sorted(partes_fecha_e[2].unique())

año_e, mes_e, dia_e


(array(['2025'], dtype=object),
 array(['06', '05', '02', '04', '03', '09', '08', '07', '01', '11', '10',
        '12'], dtype=object),
 ['01',
  '02',
  '03',
  '04',
  '05',
  '06',
  '07',
  '08',
  '09',
  '10',
  '11',
  '12',
  '13',
  '14',
  '15',
  '16',
  '17',
  '18',
  '19',
  '20',
  '21',
  '22',
  '23',
  '24',
  '25',
  '26',
  '27',
  '28',
  '29',
  '30',
  '31'])

In [14]:
#Verificar si hay valores nulos en 'started_at' y 'ended_at'

df_muestra['started_at'].isna().sum()
df_muestra['ended_at'].isna().sum()

np.int64(0)

In [15]:
#Revisamos duplicados, aunque es logico que existan duplicados en fechas de inicio y fin de viajes, pero sin que la cifra sea exagerada
#Además estos duplicados se conservan ya que los identificadores de viaje son únicos, entonces no afectan la unicidad del registro.

print(df_muestra['started_at'].duplicated().sum())
print(df_muestra['ended_at'].duplicated().sum())


0
0


### Data profiling completo

Se verificó que no hay nulos, las fechas tienen longitud y formato coherente para todos los registros de started_at y ended_at, hay duplicados válidos.

In [16]:
# Convertir fechas a formato datetime

df_muestra['started_at']=pd.to_datetime(df_muestra['started_at'], errors='coerce')
df_muestra['ended_at']=pd.to_datetime(df_muestra['ended_at'], errors='coerce')

In [17]:
#Revisar que la conversión a datetime se haya realizado correctamente y que no haya valores nulos introducidos por errores de formato.

print(df_muestra[['started_at','ended_at']].head())
print(df_muestra[['started_at','ended_at']].dtypes)
print(df_muestra['started_at'].isna().sum())
print(df_muestra['ended_at'].isna().sum())
print(df_muestra[['started_at','ended_at']].shape)

                     started_at                ended_at
2021231 2025-06-30 19:16:08.604 2025-06-30 19:21:26.124
1103055 2025-05-18 12:36:15.975 2025-05-18 12:55:40.536
1834169 2025-06-17 20:43:05.503 2025-06-17 21:04:51.961
277055  2025-02-02 05:04:52.086 2025-02-02 05:08:16.654
740603  2025-04-07 10:24:51.016 2025-04-07 10:31:27.276
started_at    datetime64[ns]
ended_at      datetime64[ns]
dtype: object
0
0
(100000, 2)


### Type Casting Completo

Las fechas fueron convertidas de string a datetime64[ns]. Además se hizo una validación rapida, confirmando nuevamente que no hay nulos, y las filas estan completas.
Aspectos a revisar duracion del viaje: que dure  un tiempo logico, no -4 minutos o 00 minutos. Duraciones extremas: minimo y maximo de fechas. Media mediana y moda


In [18]:
#Orden temporal de las fechas

orden_invalido=df_muestra['started_at'] > df_muestra['ended_at']
print(orden_invalido.sum())
df_muestra[orden_invalido].head()

2


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
5107759,083534D28DA37F72,classic_bike,2025-11-02 01:17:57.001,2025-11-02 01:05:27.752,Clark St & Grace St,CHI00301,Grace St & Central Ave,CHI01799,41.95078,-87.659172,41.949533,-87.767265,member
5333210,F87FA50B8D40FA97,electric_bike,2025-11-02 01:54:25.185,2025-11-02 01:10:50.859,NaN,NaN,Michigan Ave & Madison St,CHI01915,41.88000,-87.640000,41.882134,-87.625125,casual


In [19]:
#Calcular la duración de los viajes en minutos y verificar si hay valores negativos, lo cual indicaría un error en las fechas.

df_muestra['duration_min']=(df_muestra['ended_at'] - df_muestra['started_at']).dt.total_seconds() / 60
tiempo_negativo= df_muestra['duration_min']<0
print(tiempo_negativo.sum())    
tiempo_negativo.value_counts()

#El resultado muestra que los registros con duración negativa son 2 corresponde al orden temporal inválido que se identificó previamente, por lo que se confirma que esos registros tienen un error en las fechas de inicio y fin del viaje.

2


duration_min
False    99998
True         2
Name: count, dtype: int64

In [20]:
#Verificar si hay registros con duración de viaje igual a cero, lo cual podría indicar viajes extremadamente cortos o errores en las fechas.

tiempo_cero= df_muestra['duration_min']==0
print(tiempo_cero.sum())

0


In [21]:
#Revisar el rango de duración de los viajes para identificar posibles valores atípicos o errores en los datos.

print(df_muestra['duration_min'].min())
print(df_muestra['duration_min'].max())

-43.5721
1499.9600500000001


In [22]:
#Estoy analizando los percentiles para entender mejor la distribución de la duración de los viajes y detectar posibles valores atípicos.

df_muestra['duration_min'].describe(percentiles=[0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99])

#Definitivamente un valor de 25 horas es atipico ya que incluso el 99% de los viajes duran menos o igual a (1h 8min-88.70 min).

count    100000.000000
mean         15.978853
std          54.650981
min         -43.572100
1%            0.258716
5%            2.067148
10%           3.174220
25%           5.372471
50%           9.437192
75%          16.598817
90%          28.362220
95%          39.825555
99%          88.704491
max        1499.960050
Name: duration_min, dtype: float64

In [23]:
#Inspeccionde outliers en la duración de los viajes
outlier_2h=(df_muestra['duration_min']>2*60).sum()
outlier_6h=(df_muestra['duration_min']>6*60).sum()      
outlier_12h=(df_muestra['duration_min']>12*60).sum()
outlier_24h=(df_muestra['duration_min']>24*60).sum()

print('Outlier mayor a dos horas: ', outlier_2h)
print('Outlier mayor a seis horas: ', outlier_6h)           
print('Outlier mayor a doce horas: ', outlier_12h)
print('Outlier mayor a veinticuatro horas: ', outlier_24h)

Outlier mayor a dos horas:  561
Outlier mayor a seis horas:  168
Outlier mayor a doce horas:  131
Outlier mayor a veinticuatro horas:  100


In [24]:
#Inspección de los outliers en la duración de los viajes, Se crean columnas booleanas para identificar los outliers

df_muestra["outlier_2h"]=(df_muestra['duration_min']>2*60)
df_muestra["outlier_6h"]=(df_muestra['duration_min']>6*60)
df_muestra["outlier_12h"]=(df_muestra['duration_min']>12*60)       
df_muestra["outlier_24h"]=(df_muestra['duration_min']>24*60)
print('Outlier mayor a dos horas: ', df_muestra["outlier_2h"].sum())
print('Outlier mayor a seis horas: ', df_muestra["outlier_6h"].sum())
print('Outlier mayor a doce horas: ', df_muestra["outlier_12h"].sum())
print('Outlier mayor a veinticuatro horas: ', df_muestra["outlier_24h"].sum())

Outlier mayor a dos horas:  561
Outlier mayor a seis horas:  168
Outlier mayor a doce horas:  131
Outlier mayor a veinticuatro horas:  100


In [25]:
# Fechas fuera del rango esperado

print("Fechas maximas de started_at y ended_at: ", df_muestra['started_at'].max(), df_muestra['ended_at'].max())
print("Fechas minimas de started_at y ended_at: ", df_muestra['started_at'].min(), df_muestra['ended_at'].min())

Fechas maximas de started_at y ended_at:  2025-12-31 23:46:07.383000 2025-12-31 23:56:50.685000
Fechas minimas de started_at y ended_at:  2024-12-31 23:42:36.959000 2025-01-01 00:16:16.440000


In [26]:
# Relación entre duración del viaje y tipo de usuario

print(df_muestra.groupby("member_casual")["outlier_2h"].mean()*100)
print(df_muestra.groupby("member_casual")["outlier_6h"].mean()*100)
print(df_muestra.groupby("member_casual")["outlier_12h"].mean()*100)
print(df_muestra.groupby("member_casual")["outlier_24h"].mean()*100)


member_casual
casual    1.171724
member    0.214686
Name: outlier_2h, dtype: float64
member_casual
casual    0.326093
member    0.078353
Name: outlier_6h, dtype: float64
member_casual
casual    0.270823
member    0.051713
Name: outlier_12h, dtype: float64
member_casual
casual    0.212789
member    0.036042
Name: outlier_24h, dtype: float64


In [27]:
# Relacion entre duración del viaje y tipo de bicicleta

print(df_muestra.groupby('rideable_type')['outlier_2h'].mean()*100)
print(df_muestra.groupby('rideable_type')['outlier_6h'].mean()*100)
print(df_muestra.groupby('rideable_type')['outlier_12h'].mean()*100)
print(df_muestra.groupby('rideable_type')['outlier_24h'].mean()*100)

rideable_type
classic_bike     1.243937
electric_bike    0.192456
Name: outlier_2h, dtype: float64
rideable_type
classic_bike     0.456491
electric_bike    0.012317
Name: outlier_6h, dtype: float64
rideable_type
classic_bike     0.373752
electric_bike    0.000000
Name: outlier_12h, dtype: float64
rideable_type
classic_bike     0.285307
electric_bike    0.000000
Name: outlier_24h, dtype: float64


Se encontro lo siguiente: 2 registros que no tienen un orden valido, es decir, la hora de inicio de ruta es mayor a la hora de finalización, por lo tanto eso genera que hayan 2 duraciones negativas, no hay duraciones en 0, el promedio de viajes son 16 minutos, el 99% de los datos estan dentro del tiempo estimado de 88.70 minutos. El tiempo maximo de un viaje es de 25 horas y existen outliers de 2 horas en adelante. Al hacer la relacion con el tipo de bicicleta y el tipo de usuario se encontro que los outliers se encuentran especialmente en miembros casuales y bicicletas clasicas. Es necesario decidir qué hacer al respecto, cómo limpiarlos




### Business Rules Validation Completo

La revisión que se hizo en este paso incluye: started_at y ended_at convertidas a datetime, No nulos en fechas, Rango de fechas coherente (min/max), Regla lógica: ended_at >= started_at, Duración no negativa, Duración no extrema (outliers)

Se encontró lo siguiente:

- 2 registros con orden temporal invertido → duración negativa

- Duración máxima: 25 horas

- Los outliers se concentran en miembros casuales y bicicletas clasicas:

Outlier mayor a dos horas:  561

Outlier mayor a seis horas:  168

Outlier mayor a doce horas:  131

Outlier mayor a veinticuatro horas:  100



In [28]:
# se crean variables con los datos limpios de duración del viaje
# esto con el fin de evaluar cómo se afectarás las metricas con y sin outliers

tiempo_valido= df_muestra[df_muestra['duration_min']>=0]
tiempo_valido.shape
tiempo_valido

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_min,outlier_2h,outlier_6h,outlier_12h,outlier_24h
2021231,AC09C4230F16DC5B,electric_bike,2025-06-30 19:16:08.604,2025-06-30 19:21:26.124,Wolcott Ave & Fargo Ave,CHI01266,Elmwood Ave & Austin St,CHI00739,42.016977,-87.677725,42.025784,-87.684107,casual,5.292000,False,False,False,False
1103055,A38E8273171D67F0,electric_bike,2025-05-18 12:36:15.975,2025-05-18 12:55:40.536,NaN,NaN,Montrose Harbor,TA1308000012,41.900000,-87.620000,41.963982,-87.638181,member,19.409350,False,False,False,False
1834169,0B1F607D321397B0,electric_bike,2025-06-17 20:43:05.503,2025-06-17 21:04:51.961,NaN,NaN,Michigan Ave & Oak St,CHI00252,41.950000,-87.650000,41.900960,-87.623777,member,21.774300,False,False,False,False
277055,21649CA95A4A10B4,electric_bike,2025-02-02 05:04:52.086,2025-02-02 05:08:16.654,NaN,NaN,Clark St & Newport St,632,41.950000,-87.670000,41.944540,-87.654678,casual,3.409467,False,False,False,False
740603,D6CB6A805FC24CD2,classic_bike,2025-04-07 10:24:51.016,2025-04-07 10:31:27.276,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member,6.604333,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3212739,886300CA00A48C3E,electric_bike,2025-08-29 22:35:13.187,2025-08-29 22:46:34.353,NaN,NaN,Indiana Ave & Roosevelt Rd,CHI00450,41.860000,-87.620000,41.867888,-87.623041,casual,11.352767,False,False,False,False
4688668,C90B7D75B09195D9,electric_bike,2025-10-31 08:52:08.332,2025-10-31 08:56:15.942,Morgan Ave & 14th Pl,CHI00261,Throop St & Taylor St,CHI00389,41.862378,-87.651062,41.868968,-87.659141,member,4.126833,False,False,False,False
4958468,725DA042D4D32887,electric_bike,2025-10-14 16:27:10.291,2025-10-14 16:47:29.051,Canal St & Madison St,CHI00498,Wilton Ave & Diversey Pkwy,CHI00226,41.882409,-87.639767,41.932418,-87.652705,member,20.312667,False,False,False,False
5236138,62E7486E7CE580D1,classic_bike,2025-11-15 08:53:42.344,2025-11-15 09:13:49.922,Mies van der Rohe Way & Chicago Ave,CHI00470,Sedgwick St & Schiller St,CHI00303,41.896945,-87.621758,41.907626,-87.638566,member,20.126300,False,False,False,False


In [29]:
# Se inspecciona los datos con duracion negativa

tiempo_invalido=df_muestra[df_muestra['duration_min']<0]
tiempo_invalido.shape
tiempo_invalido

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_min,outlier_2h,outlier_6h,outlier_12h,outlier_24h
5107759,083534D28DA37F72,classic_bike,2025-11-02 01:17:57.001,2025-11-02 01:05:27.752,Clark St & Grace St,CHI00301,Grace St & Central Ave,CHI01799,41.95078,-87.659172,41.949533,-87.767265,member,-12.487483,False,False,False,False
5333210,F87FA50B8D40FA97,electric_bike,2025-11-02 01:54:25.185,2025-11-02 01:10:50.859,NaN,NaN,Michigan Ave & Madison St,CHI01915,41.88000,-87.640000,41.882134,-87.625125,casual,-43.572100,False,False,False,False


In [30]:
#Se guardan los registros con duración negativa en un archivo CSV para trazabilidad y se verifica que se hayan guardado correctamente

#tiempo_invalido.to_csv('tiempo_invalido_muestra.csv', index=False)
tiempo_invalido.shape

(2, 18)

In [31]:
#Se eliminan los registros con duración negativa para tener un conjunto de datos limpio y se verifica que se hayan eliminado correctamente

df_muestra = df_muestra[df_muestra["duration_min"] >= 0]
print(df_muestra.shape)
df_muestra.head()

(99998, 18)


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,duration_min,outlier_2h,outlier_6h,outlier_12h,outlier_24h
2021231,AC09C4230F16DC5B,electric_bike,2025-06-30 19:16:08.604,2025-06-30 19:21:26.124,Wolcott Ave & Fargo Ave,CHI01266,Elmwood Ave & Austin St,CHI00739,42.016977,-87.677725,42.025784,-87.684107,casual,5.292000,False,False,False,False
1103055,A38E8273171D67F0,electric_bike,2025-05-18 12:36:15.975,2025-05-18 12:55:40.536,NaN,NaN,Montrose Harbor,TA1308000012,41.900000,-87.620000,41.963982,-87.638181,member,19.409350,False,False,False,False
1834169,0B1F607D321397B0,electric_bike,2025-06-17 20:43:05.503,2025-06-17 21:04:51.961,NaN,NaN,Michigan Ave & Oak St,CHI00252,41.950000,-87.650000,41.900960,-87.623777,member,21.774300,False,False,False,False
277055,21649CA95A4A10B4,electric_bike,2025-02-02 05:04:52.086,2025-02-02 05:08:16.654,NaN,NaN,Clark St & Newport St,632,41.950000,-87.670000,41.944540,-87.654678,casual,3.409467,False,False,False,False
740603,D6CB6A805FC24CD2,classic_bike,2025-04-07 10:24:51.016,2025-04-07 10:31:27.276,Southport Ave & Waveland Ave,13235,Broadway & Cornelia Ave,13278,41.948226,-87.664071,41.945529,-87.646439,member,6.604333,False,False,False,False


Se eliminan los 29 datos que tienen orden temporal invertido ya que  2[tiempo negativo]/100000 [total registros]= 0.002%. 
Según las prácticas de limpieza en análisis de datos <0.1% pueden ser considerados ruido técnico y pueden ser eliminados sin tener impacto. Dado que estos registros representan eventos inválidos desde el punto de vista del negocio, fueron excluidos del dataset analítico. Las columnas temporales se conservaron, y la variable duration_min se utiliza como métrica derivada validada.

In [32]:
df_muestra['duration_min'].describe(percentiles=[0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99])

count    99998.000000
mean        15.979733
std         54.651129
min          0.003400
1%           0.258849
5%           2.067844
10%          3.174378
25%          5.373096
50%          9.437325
75%         16.599417
90%         28.362560
95%         39.825765
99%         88.705005
max       1499.960050
Name: duration_min, dtype: float64

In [33]:
# Análisis de los cuantiles y sus saltos porcentuales para saber cuales outliers tomar en la afectación de las metricas

cuantiles=df_muestra['duration_min'].quantile([0.01,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99])
saltos_cuantiles=cuantiles.pct_change()*100
saltos_cuantiles

0.01           NaN
0.05    698.862720
0.10     53.511487
0.25     69.264507
0.50     75.640363
0.75     75.891120
0.90     70.864799
0.95     40.416680
0.99    122.732709
Name: duration_min, dtype: float64

In [34]:
# Media con todos los datos

media_duracion=df_muestra['duration_min'].mean()

#Media excluyendo valores mayores a p99

p99=df_muestra['duration_min'].quantile(0.99) #primero determina el umbral del cuantil 99
sin_p99=df_muestra[df_muestra['duration_min']<=p99] #luego conservan los datos que están por debajo o igual al umbral del cuantil 99
#sin_p99

#Porcentaje que representa p99
pocentaje_p99=(df_muestra['duration_min']<=p99).mean()*100


#Media sin los outliers mayores a p99
media_sin_p99=sin_p99['duration_min'].mean()


#impacto de la media
impacto_media=(media_duracion-media_sin_p99)/media_sin_p99*100

print("Media con todos los datos: ", media_duracion)
print("Media sin los outliers del p99: ", media_sin_p99)   
print("Impacto porcentual de excluir los outliers del p99 en la media: ", round(impacto_media, 2), "%")
print("Porcentaje de datos excluidos al eliminar los outliers del p99: ", round(100-pocentaje_p99, 2), "%")

Media con todos los datos:  15.979732853990413
Media sin los outliers del p99:  12.941304694034223
Impacto porcentual de excluir los outliers del p99 en la media:  23.48 %
Porcentaje de datos excluidos al eliminar los outliers del p99:  1.0 %


In [35]:
#Mediana con todos los datos

mediana_duracion=df_muestra['duration_min'].median() 

#Mediana excluyendo valores mayores a p99

mediana_sin_p99=sin_p99['duration_min'].median()

#print

print("Mediana con todos los datos: ", mediana_duracion)
print("Mediana sin los outliers del p99: ", mediana_sin_p99)

#diferencia de la mediana con y sin outliers
diferencia_mediana=(mediana_duracion-mediana_sin_p99)/mediana_sin_p99*100
print("Diferencia porcentual de la mediana con y sin outliers del p99: ", round(diferencia_mediana, 2), "%")

#Cuán diferente es la de media de la mediana con todos los datos- Asimetría funcional

robustez_outliers=abs(media_duracion-mediana_duracion)/mediana_duracion*100

#uán diferente es la de media de la mediana con datos menores a p99- Asimetria funcional

robustez_sin_outliers=abs(media_sin_p99-mediana_sin_p99)/mediana_sin_p99*100

#print
print("Robustez con outliers: ", round(robustez_outliers, 2), "%")
print("Robustez sin outliers: ", round(robustez_sin_outliers, 2), "%")





Mediana con todos los datos:  9.437325000000001
Mediana sin los outliers del p99:  9.341841666666667
Diferencia porcentual de la mediana con y sin outliers del p99:  1.02 %
Robustez con outliers:  69.32 %
Robustez sin outliers:  38.53 %


In [36]:
import numpy as np

# Desviación estándar con todos los datos

desviacion_std_duracion=np.std(df_muestra['duration_min'])

# Desviación estándar excluyendo valores mayores a p99

desviacion_std_sin_p99=np.std(sin_p99['duration_min'])

print("Desviación estándar con todos los datos: ", desviacion_std_duracion)
print("Desviación estándar sin los outliers del p99: ", desviacion_std_sin_p99)

# Impacto de los outliers en la desviación estándar
impacto_desviacion_std=(desviacion_std_duracion-desviacion_std_sin_p99)/desviacion_std_sin_p99*100

print("Impacto porcentual de excluir los outliers del p99 en la desviación estándar: ", round(impacto_desviacion_std, 2), "%")


Desviación estándar con todos los datos:  54.65085571242025
Desviación estándar sin los outliers del p99:  11.892090183298428
Impacto porcentual de excluir los outliers del p99 en la desviación estándar:  359.56 %


In [37]:
#Calcular percentiles 99 y 95 con outliers

p95=df_muestra['duration_min'].quantile(0.95)

print(p95,p99)

#Percentiles 99 y 95 sin outliers
p95_sin_outliers=sin_p99['duration_min'].quantile(0.95)
p99_sin_outliers=sin_p99['duration_min'].quantile(0.99)

print("Percentil 95 sin outliers del p99: ", p95_sin_outliers)
print("Percentil 99 sin outliers del p99: ", p99_sin_outliers)

#Impacto de los outliers en los percentiles 95 y 99
impacto_p95=(p95-p95_sin_outliers)/p95_sin_outliers*100
impacto_p99=(p99-p99_sin_outliers)/p99_sin_outliers*100

print("Impacto porcentual de excluir los outliers del p99 en el percentil 95: ", round(impacto_p95, 2), "%")
print("Impacto porcentual de excluir los outliers del p99 en el percentil 99: ", round(impacto_p99, 2), "%")    


39.82576499999999 88.7050053333333
Percentil 95 sin outliers del p99:  36.53919583333333
Percentil 99 sin outliers del p99:  61.5383913333333
Impacto porcentual de excluir los outliers del p99 en el percentil 95:  8.99 %
Impacto porcentual de excluir los outliers del p99 en el percentil 99:  44.15 %


In [38]:
# Indice completo de impacto de los outliers en las métricas centrales (media, mediana, desviación estándar, percentiles)

indice_impacto_outliers = (impacto_media + impacto_desviacion_std + impacto_p99)/3
print("Indice completo de impacto de los outliers en las métricas centrales: ", round(indice_impacto_outliers, 2), "%")


Indice completo de impacto de los outliers en las métricas centrales:  142.39 %


In [39]:
# IQR Rango intercuartilico

Q1=df_muestra['duration_min'].quantile(0.25)    
Q3=df_muestra['duration_min'].quantile(0.75)

#Formula del IQR
IQR=Q3-Q1

round(IQR, 2)

#Limite inferior y superior para detectar outliers con el método del IQR

limite_inferior=Q1-1.5*IQR
limite_superior=Q3+1.5*IQR

limite_inferior, limite_superior
#1.5 detecta outliers leves; 3 detecta outliers extremos. Recordemos los percentiles para comparar con los limites del IQR

'''
min          0.003400
1%           0.258849
5%           2.067844
10%          3.174378
25%          5.373096
50%          9.437325
75%         16.599417
90%         28.362560-->Limite superior del IQR: 33.43
95%         39.825765
99%         88.705005
max       1499.960050
IQR inferior: -11.46
IQR superior: 33.43
El IQR nos dice que desde 33.43 minutos comienza haber un comportamiento atipico'''

#Porcentaje de viajes fuera del limite superior del IQR
viajes_fuera_iqr=(df_muestra['duration_min']>limite_superior).mean()*100
print("Porcentaje de viajes fuera del limite superior del IQR: ", round(viajes_fuera_iqr, 2), "%")

#Porcentaje de viajes dentro del limite superior del IQR
viajes_dentro_iqr=(df_muestra['duration_min']<=limite_superior).mean()*100
print("Porcentaje de viajes dentro del limite superior del IQR: ", round(viajes_dentro_iqr, 2), "%")

Porcentaje de viajes fuera del limite superior del IQR:  7.13 %
Porcentaje de viajes dentro del limite superior del IQR:  92.87 %


## Nivel 2 completo

Con base en el análisis exploratorio previo, se establecen las siguientes decisiones analíticas para la variable de duración:

- La mediana se adopta como la medida principal de tendencia central, dado que demostró ser robusta frente a la cola derecha pesada observada en la distribución.
- No se procederá a la eliminación de valores atípicos, ya que la evidencia (impacto en desviación estándar, proporción sobre el límite IQR y persistencia de la asimetría) sugiere que estos corresponden a variabilidad estructural real y no a errores de registro.
- Se incorporará el uso de percentiles operativos (p75, p90 y p95) para caracterizar la intensidad y los niveles altos de duración dentro de la distribución.
- La media se reportará únicamente como indicador complementario de asimetría y sensibilidad a la cola superior, y no como medida representativa del comportamiento típico.
  
Las columnas creadas en este nivel son duration_min y day_of_week


## Nivel 3 Columnas calculadas

El objetivo de este nivel es transformar datos crudos en variables analíticas que permitan comparar: casual vs member → comportamiento de uso y para ello se crearan nuevas columnas.

- Temporal Feature Extraction: Se crean variables derivadas del tiempo

- Behavioral Bucketing: Se agrupan usuarios o eventos en categorías específicas basadas en sus acciones, interacciones o patrones de comportamiento.

- Data Validation: Validar que los datos estan completos, sean integros y representativos

In [40]:
#Se crean columnas derivadas del tiempo para analizar patrones de comportamiento en los usuarios

df_muestra['day_of_week']=df_muestra['started_at'].dt.day_name()
df_muestra['month']=df_muestra['started_at'].dt.month_name()
df_muestra['hour']=df_muestra['started_at'].dt.hour
df_muestra['is_weekend']=df_muestra['day_of_week'].isin(['Saturday', 'Sunday'])

print(df_muestra[['day_of_week','month','hour','is_weekend']].head())

'''print(df_muestra['day_of_week'].value_counts(normalize=True)*100)
print(df_muestra['month'].value_counts(normalize=True)*100)
print(df_muestra['hour'].value_counts(normalize=True)*100)    
print(df_muestra['is_weekend'].value_counts(normalize=True)*100)

Se observa coherencia en la distribución de los días de la semana, hora, meses y fines de semana.'''



        day_of_week     month  hour  is_weekend
2021231      Monday      June    19       False
1103055      Sunday       May    12        True
1834169     Tuesday      June    20       False
277055       Sunday  February     5        True
740603       Monday     April    10       False


"print(df_muestra['day_of_week'].value_counts(normalize=True)*100)\nprint(df_muestra['month'].value_counts(normalize=True)*100)\nprint(df_muestra['hour'].value_counts(normalize=True)*100)    \nprint(df_muestra['is_weekend'].value_counts(normalize=True)*100)\n\nSe observa coherencia en la distribución de los días de la semana, hora, meses y fines de semana."

A continuación se creará una columna que nos permitirá categorizar la duración de viajes. Los rangos de duración se definen considerando la distribución percentílica de los datos y criterios de interpretabilidad de negocio, permitiendo diferenciar comportamientos de uso entre ciclistas casuales y miembros.

| Percentil | Valor real | Corte profesional |
| --------- | ---------- | ----------------- |
| p25       | 5.37       | → 5 o 10          |
| p50       | 9.43       | → 10              |
| p75       | 16.59      | → 15 o 20         |
| p90       | 28.36      | → 30              |



In [41]:
#Clasificación de los viajes en categorías de duración (corto, medio, largo) para analizar su distribución y relación con otras variables.

bins=[0, 3, 10, 20, 35, float('inf')]

labels=['Muy Corto (0-3 min)',
        'Corto (3-10 min)',
          'Medio (10-20 min)', 
          'Largo (20-35 min)',
            'Muy Largo (35+ min)']

df_muestra['ride_length']=pd.cut(df_muestra['duration_min'], bins=bins, labels=labels, right=True, include_lowest=True)

In [42]:
print(df_muestra['ride_length'].value_counts())
print(df_muestra['ride_length'].isna().sum())
print(round (df_muestra['ride_length'].value_counts(normalize=True)*100, 2))

ride_length
Corto (3-10 min)       43757
Medio (10-20 min)      28387
Largo (20-35 min)      12294
Muy Corto (0-3 min)     9033
Muy Largo (35+ min)     6527
Name: count, dtype: int64
0
ride_length
Corto (3-10 min)       43.76
Medio (10-20 min)      28.39
Largo (20-35 min)      12.29
Muy Corto (0-3 min)     9.03
Muy Largo (35+ min)     6.53
Name: proportion, dtype: float64


In [43]:
#Validacion cruzada entre duración de viaje y tipo de usuario

print(df_muestra.groupby("member_casual")["ride_length"].value_counts(normalize=True)*100)

member_casual  ride_length        
casual         Corto (3-10 min)       35.912671
               Medio (10-20 min)      28.727373
               Largo (20-35 min)      15.464972
               Muy Largo (35+ min)    11.709272
               Muy Corto (0-3 min)     8.185712
member         Corto (3-10 min)       48.206478
               Medio (10-20 min)      28.194882
               Largo (20-35 min)      10.496294
               Muy Corto (0-3 min)     9.513735
               Muy Largo (35+ min)     3.588610
Name: proportion, dtype: float64


In [44]:
#categorizar los viajes en función de la hora del día (mañana, tarde, noche) para analizar patrones de uso a lo largo del día.
bins_hora=[0, 6, 12, 18, 24]

labels_hora=['Madrugada (0-6)', 
             'Mañana (6-12)', 
             'Tarde (12-18)', 
             'Noche (18-24)']

df_muestra["time_of_day"]=pd.cut(df_muestra['hour'], bins=bins_hora, labels=labels_hora, right=False, include_lowest=True)

print(round(df_muestra['time_of_day'].value_counts(normalize=True)*100, 2))

time_of_day
Tarde (12-18)      43.96
Mañana (6-12)      26.11
Noche (18-24)      25.98
Madrugada (0-6)     3.95
Name: proportion, dtype: float64


In [45]:
season={
    'December': 'Invierno',
    'January': 'Invierno',
    'February': 'Invierno',
    'March': 'Primavera',
    'April': 'Primavera',
    'May': 'Primavera',
    'June': 'Verano',
    'July': 'Verano',
    'August': 'Verano',
    'September': 'Otoño',
    'October': 'Otoño',
    'November': 'Otoño'
}

df_muestra['season']=df_muestra['month'].map(season)

df_muestra['season'].head()

2021231       Verano
1103055    Primavera
1834169       Verano
277055      Invierno
740603     Primavera
Name: season, dtype: object

In [46]:
col_nuevas=['day_of_week','month','hour','is_weekend','ride_length','time_of_day']
df_muestra[col_nuevas].isna().sum()

day_of_week    0
month          0
hour           0
is_weekend     0
ride_length    0
time_of_day    0
dtype: int64

 ### Nivel 3 Completo
 
 Se crearon las siguientes variables: ride_length, hour, day_of_week, month, is_weekend, time_of_day; estas variables permiten identificar patrones de uso por momento del día, día de la semana y duración del viaje, facilitando la comparación entre ciclistas ocasionales y miembros anuales.. Se verificó la coherencia de rangos y la ausencia de valores nulos en las variables derivadas. Las distribuciones fueron revisadas para asegurar consistencia lógica antes de continuar con el análisis.

## Nivel 4: Validación de Variables Categórica

Las variables categóricas representan atributos cualitativos clave del comportamiento del usuario, member_casual y rideable_type . En el contexto de este proyecto, estas variables son fundamentales porque permiten segmentar la población y comparar patrones de uso entre ciclistas ocasionales y miembros anuales.

A diferencia de las variables temporales (analizadas en el Nivel 3), que describen cuándo y cuánto se usa el servicio, las variables categóricas permiten entender quién usa el servicio y cómo lo utiliza, lo cual es crítico para diseñar estrategias de marketing orientadas a la conversión de usuarios. 

Las variables categóricas se abordan después de la ingeniería temporal porque:
- La duración y el tiempo ya están validados, lo que permite interpretar correctamente el comportamiento.
- Estas variables no requieren transformación matemática compleja, pero sí validación semántica rigurosa.
- Son la base del análisis comparativo de negocio, por lo que deben estar completamente limpias antes de cruzarlas con métricas temporales.

Las fases de este nivel son las siguientes:

| Etapa                   | Rol                    |
| ----------------------- | ---------------------- |
| Ingeniería de variables | crea señales           |
| Crosstab                | muestra distribuciones |
| Puntos porcentuales     | cuantifican la brecha  |
| Ranking                 | prioriza acción        |


In [47]:
#Data profiling; se revisa la consistencia semántica delas variables categoricas

print(df_muestra['member_casual'].value_counts().sum())
print(df_muestra['rideable_type'].value_counts().sum())
print(df_muestra[['member_casual', 'rideable_type']].nunique())
print(df_muestra['rideable_type'].unique())
print(df_muestra['member_casual'].unique())
print(df_muestra['member_casual'].value_counts())
print(df_muestra['rideable_type'].value_counts())

# No se observan inconsistencias semánticas en las categorías.



99998
99998
member_casual    2
rideable_type    2
dtype: int64
['electric_bike' 'classic_bike']
['casual' 'member']
member_casual
member    63813
casual    36185
Name: count, dtype: int64
rideable_type
electric_bike    64949
classic_bike     35049
Name: count, dtype: int64


In [48]:
# Se verifico que no hay valores nulos, sin embargo, se revisa explicitamente para documentar formalmente

print(df_muestra['member_casual'].isna().sum())
print(df_muestra['rideable_type'].isna().sum())

0
0


In [49]:
print(round (df_muestra['member_casual'].value_counts(normalize=True)*100), 2)
print(round (df_muestra['rideable_type'].value_counts(normalize=True)*100), 2)

member_casual
member    64.0
casual    36.0
Name: proportion, dtype: float64 2
rideable_type
electric_bike    65.0
classic_bike     35.0
Name: proportion, dtype: float64 2


In [50]:
#Cross validation, Cruzamos la infromación de memeber_casual y rideable_bike para conocer las preferecnias de uso por tipo de usuario

tabla_mxr=pd.crosstab(df_muestra['member_casual'], 
            df_muestra['rideable_type'],
            normalize="index")*100

print(tabla_mxr)
#print(tabla.sum(axis=1))

'''cross_pct = (
    df_muestra
    .groupby("member_casual")["rideable_type"]
    .value_counts(normalize=True)
    .mul(100)
    .rename("percentage")
    .reset_index()
)

cross_pct-->Versión escalable'''

rideable_type  classic_bike  electric_bike
member_casual                             
casual            33.826171      66.173829
member            35.743501      64.256499


'cross_pct = (\n    df_muestra\n    .groupby("member_casual")["rideable_type"]\n    .value_counts(normalize=True)\n    .mul(100)\n    .rename("percentage")\n    .reset_index()\n)\n\ncross_pct-->Versión escalable'

In [51]:
# Se calcula la diferencia de puntos porcentuales para comprobar si hay un patrón de comportamiento en el uso de bicicletas

dif_ppmxr = tabla_mxr.loc["casual"] - tabla_mxr.loc["member"]
dif_ppmxr


#dif_pp diferencia de puntos porcentuales

rideable_type
classic_bike    -1.917329
electric_bike    1.917329
dtype: float64

Ambos usuarios prefieren las electricas sin embargo al hacer la diferencia de puntos porcentuales. es muy baja, esto nos indica que el tipo de bicicleta NO discrimina bien entre casual y member

Diferencia (pp)	Interpretación

< 3 pp	prácticamente igual

3–7 pp	diferencia leve

7–15 pp	diferencia moderada

15 pp	diferencia fuerte

positivo → más casuales
negativo → más miembros

Aunque ambos tipos de usuarios muestran preferencia por bicicletas eléctricas, la diferencia entre miembros y casuales es menor a 2 puntos porcentuales, lo que indica que el tipo de bicicleta no constituye un factor diferenciador relevante en el comportamiento de uso- En los usuarios sí se nota una distinción entre eléctricas y clásicas, pero el tipo de bicicleta no distingue entre miembros y casuales-.

In [52]:
#Tabla cruzada member_casual y hour

tabla_mxh=pd.crosstab(df_muestra['member_casual'],
                      df_muestra['hour'],
                      normalize="index")*100

#print(tabla_mxh)
tabla_mxh

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
member_casual,,,,,,,,,,,,,,,,,,,,,
casual,1.906867,1.279536,0.773801,0.420064,0.373083,0.514025,1.274009,2.346276,3.631339,3.634103,...,6.895122,7.887246,8.854498,9.564737,8.558795,6.137902,4.438303,3.708719,3.114550,2.235733
member,0.933979,0.532807,0.346324,0.216257,0.249166,0.924576,2.781565,5.722972,7.227367,4.923762,...,5.260684,6.656951,9.364863,10.786203,8.236566,5.755880,3.927099,3.170200,2.233087,1.407237


In [53]:
dif_ppmxh = tabla_mxh.loc["casual"] - tabla_mxh.loc["member"]
dif_ppmxh_abs = dif_ppmxh.abs().sort_values(ascending=False)
dif_ppmxh_abs.head(10)

hour
8     3.596029
7     3.376696
14    1.634439
6     1.507556
9     1.289659
15    1.230295
17    1.221467
13    1.150326
12    1.003052
0     0.972889
dtype: float64

La variable hour presenta diferencias leves entre miembros y usuarios casuales, concentradas en las horas pico de la mañana (7–8 a. m.). Sin embargo, la magnitud de la diferencia (<4 pp) indica un bajo poder discriminatorio, por lo que la hora del día, de forma aislada, no constituye una palanca fuerte para estrategias de conversión.

In [54]:
tabla_mxt=pd.crosstab(df_muestra['member_casual'],
                      df_muestra['time_of_day'],
                      normalize='index')*100

dif_ppmxt=tabla_mxt.loc['casual']-tabla_mxt.loc['member']
dif_ppmxt

time_of_day
Madrugada (0-6)    2.064267
Mañana (6-12)     -8.814480
Tarde (12-18)      3.286280
Noche (18-24)      3.463932
dtype: float64

Los miembros muestran un patrón más utilitario concentrado en la mañana, mientras que los usuarios casuales presentan mayor presencia relativa en franjas recreativas (tarde y noche).

In [55]:
##Tabla cruzada member_casual y ride_length

tabla_mxrl=pd.crosstab(df_muestra['member_casual'],
                      df_muestra['ride_length'],
                      normalize='index')*100

dif_ppmxrl=tabla_mxrl.loc['casual']-tabla_mxrl.loc['member']
dif_ppmxrl

ride_length
Muy Corto (0-3 min)    -1.328023
Corto (3-10 min)      -12.293807
Medio (10-20 min)       0.532491
Largo (20-35 min)       4.968678
Muy Largo (35+ min)     8.120661
dtype: float64

Los miembros utilizan la bicicleta principalmente para trayectos cortos y funcionales, mientras que los usuarios casuales se inclinan hacia viajes largos de carácter recreativo.

In [56]:
tabla_mxm=pd.crosstab(df_muestra['member_casual'],
                      df_muestra['month'],
                      normalize='index')*100
dif_ppmxm=tabla_mxm.loc['casual']-tabla_mxm.loc['member']
dif_ppmxm

month
April       -2.023796
August       4.069974
December    -1.712282
February    -2.168217
January     -2.047881
July         4.352778
June         3.375927
March       -1.558357
May          0.285744
November    -2.419358
October     -0.685328
September    0.530797
dtype: float64

Los usuarios casuales aumentan en meses cálidos, mientras que los miembros mantienen mayor presencia relativa en meses fríos. Crear estaciones para verlo a mayor detalle

In [57]:
tabla_mxs=pd.crosstab(df_muestra['member_casual'],
                      df_muestra['season'],
                      normalize='index')*100

dif_ppmxs=tabla_mxs.loc['casual']-tabla_mxs.loc['member']
dif_ppmxs

season
Invierno     -5.928380
Otoño        -2.573889
Primavera    -3.296410
Verano       11.798679
dtype: float64

La estacionalidad muestra un patrón claro de uso recreativo. Durante el verano se observa una sobrerrepresentación significativa de usuarios casuales (+11.8 pp), lo que sugiere que esta estación concentra oportunidades clave de conversión hacia membresías anuales.

In [58]:
tabla_mxw=pd.crosstab(df_muestra['member_casual'],
                      df_muestra['day_of_week'],
                      normalize='index')*100
dif_ppmxw=tabla_mxw.loc['casual']-tabla_mxw.loc['member']
dif_ppmxw

day_of_week
Friday       0.953814
Monday      -2.852619
Saturday     7.811221
Sunday       5.870970
Thursday    -3.326505
Tuesday     -4.389340
Wednesday   -4.067539
dtype: float64

Los miembros muestran un patrón de uso más concentrado en días laborales, mientras que los usuarios casuales presentan mayor actividad durante los fines de semana.

In [59]:
tabla_mxiw=pd.crosstab(df_muestra['member_casual'],
                       df_muestra['is_weekend'],
                       normalize='index')*100
dif_ppmxiw=tabla_mxiw.loc['casual']-tabla_mxiw.loc['member']
dif_ppmxiw

is_weekend
False   -13.682191
True     13.682191
dtype: float64

Las variables analizadas muestran un patrón conductual consistente donde el uso en fines de semana y los trayectos de mayor duración se asocian positivamente con el segmento casual, mientras que los trayectos cortos en días laborales se relacionan con el segmento de miembros

In [ ]:
impactos = {
    "rideable_type": dif_ppmxr.abs().max(),
    "hour": dif_ppmxh.abs().max(),
    "time_of_day": dif_ppmxt.abs().max(),
    "ride_length": dif_ppmxrl.abs().max(),
    "day_of_week": dif_ppmxw.abs().max(),
    "is_weekend": dif_ppmxiw.abs().max(),
    "month": dif_ppmxm.abs().max(),
    "season": dif_ppmxs.abs().max()
}

impactos

{'rideable_type': np.float64(1.9173294752836156),
 'hour': np.float64(3.5960285117051103),
 'time_of_day': np.float64(8.814479589920182),
 'ride_length': np.float64(12.293807307524723),
 'day_of_week': np.float64(7.811221185495402),
 'is_weekend': np.float64(13.682190800686133),
 'month': np.float64(4.352777689196069),
 'season': np.float64(11.798679089632493)}

In [61]:
ranking = (
    pd.Series(impactos, name="impacto_pp")
    .sort_values(ascending=False)
    .to_frame()
)

ranking

,impacto_pp
is_weekend,13.682191
ride_length,12.293807
season,11.798679
time_of_day,8.814480
day_of_week,7.811221
month,4.352778
hour,3.596029
rideable_type,1.917329


| Variable      | Impacto | Lectura     |
| ------------- | ------- | ----------- |
| is_weekend    | 13.68   | 🔥 Fuerte   |
| ride_length   | 12.29   | 🔥 Fuerte   |
| season        | 11.80   | 🔥 Fuerte   |
| time_of_day   | 8.81    | ⚠️ Moderado |
| day_of_week   | 7.81    | ⚠️ Moderado |
| rideable_type | 1.91    | ❄️ Débil    |


### Nivel 5

En esta fase se evalúan las variables geográficas y de estaciones con el objetivo de: verificar la calidad y consistencia espacial de los datos; identificar posibles patrones de uso por ubicación; detectar zonas o estaciones con comportamiento diferencial entre usuarios casuales y miembros. Este nivel es clave porque añade la dimensión “dónde ocurre el viaje”, complementando el análisis temporal realizado previamente.

Variables:

- start_lat
- start_lng
- end_lat
- end_lng

1. Perfilamiento de coordenadas: Detectar la integridad básica de las variables de coordenadas, nulos, valores en cero y atípicos
2. Ingenieria de variables geograficas: Proceso de transformar coordenadas crudas (lat/lng) en variables analíticas que permitan entender el comportamiento espacial de los usuarios.

##### Nota importante para entender este punto:

#### *Latitud → norte ↕ sur*

Mide qué tan arriba o abajo estás respecto al ecuador. Va de -90 a +90

| Latitud  | Significado      |
| -------- | ---------------- |
| positiva | hemisferio norte |
| negativa | hemisferio sur   |

(~41.9) = Chicago → hemisferio norte.

#### *Longitud → oeste ↔ este*

Mide qué tan a la izquierda o derecha estás respecto al meridiano de Greenwich. Va de -180 a +180

| Longitud | Significado |
| -------- | ----------- |
| negativa | oeste       |
| positiva | este        |


(~ -87) = oeste de Greenwich.

In [62]:
df_muestra[['end_lat', 'start_lat', 'end_lng', 'start_lng']].isna().sum()

end_lat      98
start_lat     0
end_lng      98
start_lng     0
dtype: int64

In [63]:
#Por error puse todo end_lat en cero entonces ejecute este codigo para recuperar los datos
#df_backup = pd.read_csv('muestra_ciclistas.csv')

# restaurar la columna dañada
#df_muestra['end_lat'] = df_backup['end_lat']


In [64]:
ceros=(df_muestra[['end_lat', 'start_lat', 'end_lng', 'start_lng']]==0).sum()
ceros


end_lat      0
start_lat    0
end_lng      0
start_lng    0
dtype: int64

In [65]:
'''df_muestra = df_muestra.merge(
    df_backup[['ride_id', 'end_lat', 'end_lng']],
    on='ride_id',
    how='left'
)'''

"df_muestra = df_muestra.merge(\n    df_backup[['ride_id', 'end_lat', 'end_lng']],\n    on='ride_id',\n    how='left'\n)"

In [66]:
# conservar una sola versión
'''df_muestra['end_lat'] = df_muestra['end_lat_x']
df_muestra['end_lng'] = df_muestra['end_lng_x']

# eliminar columnas duplicadas
df_muestra.drop(columns=[
    'end_lat_x','end_lat_y',
    'end_lng_x','end_lng_y'
], inplace=True)'''

"df_muestra['end_lat'] = df_muestra['end_lat_x']\ndf_muestra['end_lng'] = df_muestra['end_lng_x']\n\n# eliminar columnas duplicadas\ndf_muestra.drop(columns=[\n    'end_lat_x','end_lat_y',\n    'end_lng_x','end_lng_y'\n], inplace=True)"

In [67]:
df_muestra[['end_lat','end_lng','start_lat','start_lng']].isna().sum()


end_lat      98
end_lng      98
start_lat     0
start_lng     0
dtype: int64

In [68]:
df_muestra[['end_lat','end_lng','start_lat','start_lng']].describe()

,end_lat,end_lng,start_lat,start_lng
count,99900.000000,99900.000000,99998.000000,99998.000000
mean,41.904303,-87.646850,41.903840,-87.646509
std,0.044589,0.027371,0.044411,0.027295
min,41.650000,-87.841098,41.651903,-87.843960
25%,41.882409,-87.660000,41.882060,-87.660000
50%,41.899930,-87.642884,41.898587,-87.641736
75%,41.930000,-87.630000,41.930000,-87.629926
max,42.210000,-87.528232,42.070000,-87.528232


Se valido la integridad de los datos, los nulos representan en 0.098% de los datos por lo que no representa un problema estructural y se decide usar la variable para hacer data engineering. Se crea distance_km  con *Haversine*; Una fórmula matemática que calcula la distancia real entre dos puntos en la superficie de la Tierra usando latitud y longitud.

| Variable  | Significado |
| --------- | ----------- |
| start_lat | lat1        |
| start_lng | lon1        |
| end_lat   | lat2        |
| end_lng   | lon2        |


In [69]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371  # radio Tierra en km-->estandar
    
    lat1_rad = np.radians(lat1) #Convertir grados a radianes
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2)
    lon2_rad = np.radians(lon2)
    
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad)*np.cos(lat2_rad)*np.sin(dlon/2)**2 #-->formula de Haversive
    c = 2 * np.arcsin(np.sqrt(a))
    
    return R * c # distancia = radio × ángulo

# crear distancia solo donde hay coordenadas completas
mask_valid = df_muestra['end_lat'].notna()

df_muestra.loc[mask_valid, 'distance_km'] = haversine_vectorized(
    df_muestra.loc[mask_valid, 'start_lat'],
    df_muestra.loc[mask_valid, 'start_lng'],
    df_muestra.loc[mask_valid, 'end_lat'],
    df_muestra.loc[mask_valid, 'end_lng']
)

In [89]:
df_muestra['distance_km'].dtype

CategoricalDtype(categories=['Muy corto: 0-1km', 'Corto: 1-3km', 'Medio: 3-6km',
                  'Largo: 6-15km', 'Muy largo: 15+km'],
, ordered=True, categories_dtype=object)

In [70]:
df_muestra['distance_km'].describe()

count    99900.000000
mean         2.206325
std          1.994350
min          0.000000
25%          0.909497
50%          1.614191
75%          2.884720
max         25.239582
Name: distance_km, dtype: float64

In [71]:
(df_muestra['distance_km']==0).mean()*100

np.float64(6.120122402448049)

6 de cada 100 viajes marcan 0 distancia 
| % de ceros | Nivel de alerta | Interpretación              | Acción recomendada   |
| ---------- | --------------- | --------------------------- | -------------------- |
| < 1%       | 🟢 Bajo         | Ruido normal del sistema    | Mantener             |
| 1% – 5%    | 🟡 Moderado     | Revisar contexto            | Mantener con nota    |
| 5% – 10%   | 🟠 Alto         | Posible mezcla real + ruido | Investigar (tu caso) |
| > 10%      | 🔴 Crítico      | Probable problema de datos  | Depurar              |


In [72]:
df_muestra.loc[df_muestra["distance_km"] == 0, "duration_min"].describe()

count    6120.000000
mean       18.422581
std        51.593274
min         0.003400
25%         0.493258
50%         3.031675
75%        19.718683
max      1312.257367
Name: duration_min, dtype: float64

In [73]:
'''df_muestra = df_muestra.merge(
    df_backup[['ride_id', 'end_lat', 'end_lng']],
    on='ride_id',
    how='left'
)'''

"df_muestra = df_muestra.merge(\n    df_backup[['ride_id', 'end_lat', 'end_lng']],\n    on='ride_id',\n    how='left'\n)"

In [74]:
df_muestra["speed_kmh"] = (df_muestra["distance_km"] /(df_muestra["duration_min"] / 60))

In [75]:
(df_muestra["distance_km"] == 0).sum() 


np.int64(6120)

In [76]:
df_muestra.loc[
    (df_muestra["distance_km"] == 0) &
    (df_muestra["duration_min"] > 60)
].shape[0]

469

In [77]:
df_muestra['speed_kmh'].describe()

count    99900.000000
mean        12.178425
std         25.704421
min          0.000000
25%          8.107963
50%         11.576942
75%         15.135393
max       3990.828379
Name: speed_kmh, dtype: float64

In [78]:
(df_muestra["speed_kmh"] > 40).sum()


np.int64(452)

In [79]:
(df_muestra["speed_kmh"] > 60).sum()

np.int64(344)

La variable speed_kmh indica que la tendencia central (media y mediana) se encuentra dentro del rango esperado para bicicletas urbanas.La distribución principal es coherente con velocidades típicas de uso recreativo y utilitario, sin embargo, el valor máximo evidencia la presencia de outliers extremos, físicamente imposibles. Por ello, se indaga en la proporción de valores atípicos, es menor al 1% del dataset, lo que indica:
- Los outliers no distorsionan la distribución central
- El dataset sigue siendo confiable para análisis de comportamiento

Por lo tanto se decide mantener los registros y se documenta esta anomalia:

Registros > 40 km/h: 452 casos (~0.45%)

Registros > 60 km/h: 344 casos (~0.34%)

seguimos en la ingenieria de variables, para ello se convierte las variables de distancia y velocidad en categorias que sean interpretables y utiles para el análisis. Granudalidad

In [80]:
bins_dist=[0,1,3,6,15,np.inf]

labels_dist=['Muy corto: 0-1km',
             'Corto: 1-3km',
             'Medio: 3-6km',
             'Largo: 6-15km',
             'Muy largo: 15+km']

df_muestra['distance_km']= pd.cut(df_muestra['distance_km'],bins=bins_dist, labels=labels_dist, include_lowest=True)

In [81]:
df_muestra['distance_km']

2021231        Corto: 1-3km
1103055       Largo: 6-15km
1834169        Medio: 3-6km
277055         Corto: 1-3km
740603         Corto: 1-3km
                 ...       
3212739    Muy corto: 0-1km
4688668    Muy corto: 0-1km
4958468        Medio: 3-6km
5236138        Corto: 1-3km
3939496        Medio: 3-6km
Name: distance_km, Length: 99998, dtype: category
Categories (5, object): ['Muy corto: 0-1km' < 'Corto: 1-3km' < 'Medio: 3-6km' < 'Largo: 6-15km' < 'Muy largo: 15+km']

In [83]:
tabla_mxd=pd.crosstab(df_muestra['member_casual'],
                      df_muestra['distance_km'],
                      normalize='index')*100

dif_ppmxd=tabla_mxd.loc['casual']-tabla_mxd.loc['member']
dif_ppmxd


distance_km
Muy corto: 0-1km    0.531545
Corto: 1-3km        1.289723
Medio: 3-6km       -1.501539
Largo: 6-15km      -0.328669
Muy largo: 15+km    0.008940
dtype: float64

La distancia recorrida no muestra diferencias sustanciales entre usuarios casuales y miembros. Los puntos porcentuales se mantienen por debajo de ±2 pp en todas las categorías, lo que indica una baja capacidad discriminativa de esta variable para segmentar el comportamiento de los usuarios.

In [84]:
impactos = {
    "rideable_type": dif_ppmxr.abs().max(),
    "hour": dif_ppmxh.abs().max(),
    "time_of_day": dif_ppmxt.abs().max(),
    "ride_length": dif_ppmxrl.abs().max(),
    "day_of_week": dif_ppmxw.abs().max(),
    "is_weekend": dif_ppmxiw.abs().max(),
    "month": dif_ppmxm.abs().max(),
    "season": dif_ppmxs.abs().max(),
    "distance_km": dif_ppmxd.abs().max()
}

impactos

ranking = (
    pd.Series(impactos, name="impacto_pp")
    .sort_values(ascending=False)
    .to_frame()
)

ranking

,impacto_pp
is_weekend,13.682191
ride_length,12.293807
season,11.798679
time_of_day,8.814480
day_of_week,7.811221
month,4.352778
hour,3.596029
rideable_type,1.917329
distance_km,1.501539
